In [0]:
confi_api_key=''
confi_api_secret=''
confi_bootstrap=''

In [0]:
order_df1=spark.readStream.format('kafka').option('kafka.bootstrap.servers',confi_bootstrap).option('kafka.security.protocol','SASL_SSL').option('kafka.sasl.mechanism','PLAIN').option("kafka.sasl.jaas.config", "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username='{}' password='{}';".format(confi_api_key, confi_api_secret))\
.option("kafka.ssl.endpoint.identification.algorithm","https") \
.option('subscribe','Orders_raw_data')\
.option("startingTimestamp",1) \
.option("maxOffsetsPerTrigger",50) \
.load()


In [0]:
from pyspark.sql.functions import *
order_schema='order_id string,user_id string,product_id string,amount int,payment_mode string,country string,order_time string'
order_df1=order_df1.select(col('key').cast('string'),col('value').cast('string'),col('topic'))
order_df1=order_df1.select(col('key').cast('string').alias('key'),from_json('value',order_schema).alias('value'))

key,value
o7,"List(o7, u7, p107, 5200, CARD, IN, 2025-01-01T10:01:35)"
o10,"List(o10, u10, p110, 7500, CARD, IN, 2025-01-01T10:02:20)"
o11,"List(o11, u1, p101, 1300, UPI, IN, 2025-01-01T10:02:35)"
o12,"List(o12, u2, p102, 4700, CARD, IN, 2025-01-01T10:02:50)"
o14,"List(o14, u4, p104, 6100, NETBANKING, IN, 2025-01-01T10:03:20)"
o19,"List(o19, u9, p109, 2100, UPI, US, 2025-01-01T10:04:35)"
o20,"List(o20, u10, p110, 7600, CARD, IN, 2025-01-01T10:04:50)"
o25,"List(o25, u15, p115, 500, UPI, IN, 2025-01-01T10:06:05)"
o26,"List(o26, u16, p116, 6200, NETBANKING, US, 2025-01-01T10:06:20)"
o28,"List(o28, u18, p118, 8900, CARD, IN, 2025-01-01T10:06:50)"


In [0]:
order_df2=order_df1.withColumn('order_type',when(col('value.amount')>6000,'HIGH_VALUE').otherwise(when((col('value.amount')>1000) & (col('value.amount')<=6000) ,'MEDIUM_VALUE').otherwise('LOW_VALUE')))
order_df2=order_df2.withColumn('is_international',when(col('value.country')=='IN',True).otherwise(False))
order_df2=order_df2.withColumn('processed_time',current_timestamp())
order_df3=order_df2.select(col('key').cast('string').alias('key'),to_json(struct(col('value.order_id').alias('order_id'),col('value.user_id').alias('user_id'),col('value.product_id').alias('product_id'),col('value.amount').alias('amount'),col('value.payment_mode').alias('payment_mode'),col('value.country').alias('country'),col('value.order_time').alias('order_time'),col('order_type'),col('is_international'),col('processed_time'))).cast('string').alias('value'))

In [0]:
order_df3.writeStream.format('kafka').option('checkpointLocation','abfss://data@storageaccountfda.dfs.core.windows.net/catalog/purple/checkpoint2').outputMode('append').trigger(availableNow=True)\
.option("kafka.bootstrap.servers",'pkc-921jm.us-east-2.aws.confluent.cloud:9092') \
.option("kafka.security.protocol","SASL_SSL") \
.option("kafka.sasl.mechanism","PLAIN") \
.option("kafka.sasl.jaas.config", "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username='{}' password='{}';".format(confi_api_key,confi_api_secret)) \
.option("kafka.ssl.endpoint.identification.algorithm","https") \
.option("topic",'processed_retail_data') \
.start()

In [0]:
order__processed_df=spark.readStream.format('kafka').option('kafka.bootstrap.servers',confi_bootstrap).option('kafka.security.protocol','SASL_SSL').option('kafka.sasl.mechanism','PLAIN').option("kafka.sasl.jaas.config", "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username='{}' password='{}';".format(confi_api_key, confi_api_secret))\
.option("kafka.ssl.endpoint.identification.algorithm","https") \
.option('subscribe','processed_retail_data')\
.option("startingTimestamp",1) \
.option("maxOffsetsPerTrigger",50) \
.load()


In [0]:
order_schema='order_id string,user_id string,product_id string,amount int,payment_mode string,country string,order_time string,order_type string,is_international boolean,processed_time timestamp'
order_processed_df1=order__processed_df.select(col('key').cast('string').alias('key'),col('value').cast('string').alias('value'))
order_processed_df1=order_processed_df1.select('key',from_json(col('value'),order_schema).alias('value'))
order_high_value_df=order_processed_df1.filter(col('value.order_type')=='HIGH_VALUE')
order_high_value_df=order_high_value_df.select(col('key').cast('string'),to_json(struct(col('value.order_id').alias('order_id'),col('value.user_id').alias('user_id'),col('value.product_id').alias('product_id'),col('value.amount').alias('amount'),col('value.payment_mode').alias('payment_mode'),col('value.country').alias('country'),col('value.order_time').alias('order_time'),col('value.order_type').alias('order_type'),col('value.is_international').alias('is_international'),col('value.processed_time').alias('processed_time'))).alias('value'))


In [0]:
order_high_value_df.writeStream.format('kafka').option('checkpointLocation','abfss://data@storageaccountfda.dfs.core.windows.net/catalog/aggregated/checkpoint2').outputMode('append').trigger(availableNow=True)\
.option("kafka.bootstrap.servers",'pkc-921jm.us-east-2.aws.confluent.cloud:9092') \
.option("kafka.security.protocol","SASL_SSL") \
.option("kafka.sasl.mechanism","PLAIN") \
.option("kafka.sasl.jaas.config", "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username='{}' password='{}';".format(confi_api_key,confi_api_secret)) \
.option("kafka.ssl.endpoint.identification.algorithm","https") \
.option("topic",'high_value_orders') \
.start()

In [0]:
high_value_df=spark.readStream.format('kafka').option('kafka.bootstrap.servers',confi_bootstrap).option('kafka.security.protocol','SASL_SSL').option('kafka.sasl.mechanism','PLAIN').option("kafka.sasl.jaas.config", "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username='{}' password='{}';".format(confi_api_key, confi_api_secret))\
.option("kafka.ssl.endpoint.identification.algorithm","https") \
.option('subscribe','high_value_orders')\
.option("startingTimestamp",1) \
.option("maxOffsetsPerTrigger",50) \
.load()

In [0]:
from pyspark.sql.functions import *
order_schema='order_id string,user_id string,product_id string,amount int,payment_mode string,country string,order_time string,order_type string,is_international boolean,processed_time timestamp'
high_value_df=high_value_df.select(col('key').cast('string'),col('value').cast('string'))
high_value_df=high_value_df.select('key',from_json(col('value'),order_schema).alias('value'))

In [0]:
orders_payment_mode_df=high_value_df.groupBy(col('value.payment_mode')).agg(count('*').alias('total_count'))
orders_payment_mode_df=orders_payment_mode_df.select(col('payment_mode').alias('key'),to_json(struct(col('payment_mode'),col('total_count'))).alias('value'))

In [0]:
orders_payment_mode_df.writeStream.format('kafka').option('checkpointLocation','abfss://data@storageaccountfda.dfs.core.windows.net/catalog/aggregated/checkpoint3').outputMode('update').trigger(availableNow=True)\
.option("kafka.bootstrap.servers",'pkc-921jm.us-east-2.aws.confluent.cloud:9092') \
.option("kafka.security.protocol","SASL_SSL") \
.option("kafka.sasl.mechanism","PLAIN") \
.option("kafka.sasl.jaas.config", "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username='{}' password='{}';".format(confi_api_key,confi_api_secret)) \
.option("kafka.ssl.endpoint.identification.algorithm","https") \
.option("topic",'count_per_payment_method') \
.start()